# Aula 10 — Regressão Linear, Logística e Avaliação de Modelos
**Ciência de Dados · Univassouras · Prof. Mesc. Diego Ramos Inácio**

---

> **Conexão com a Aula 09:** na aula anterior treinamos os primeiros modelos preditivos (Regressão Linear, Logística, Árvore, Random Forest) e calculamos métricas básicas.
> Agora aprofundamos: como **diagnosticar** se o modelo realmente funciona? Como **comparar** modelos de forma justa?
> E como evitar a armadilha do ***overfitting***?

| # | Conteúdo |
|---|----------|
| 0 | Configuração do ambiente |
| 1 | **Exemplo 1** — Regressão Linear: diagnóstico de resíduos |
| 2 | **Exemplo 2** — Regressão Logística: ROC, AUC e escolha do limiar |
| 3 | **Exemplo 3** — Validação Cruzada: avaliação justa e robusta |
| 4 | **Exemplo 4** — Overfitting vs Underfitting na prática |
| 5 | Comparativo final de modelos |
| 6 | **Exercício** — sua vez! |

> **Dica:** execute cada célula com `Shift + Enter`.

## 0 · Configuração do Ambiente

Instale as dependências caso ainda não tenha (remova o `#` e execute).

In [ ]:
# !pip install numpy pandas matplotlib scikit-learn scipy --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import (
    train_test_split, cross_val_score,
    learning_curve, StratifiedKFold
)
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    roc_curve, roc_auc_score
)
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
print('Ambiente pronto!')

---
## 1 · Exemplo 1 — Regressão Linear: Diagnóstico de Resíduos
### Problema: prever o tempo de entrega de encomendas

**Variável alvo (Y):** Tempo de entrega (horas)  
**Preditores (X):** Distância (km), Peso (kg), Paradas intermediárias  
**Modelo:** Ŷ = β₀ + β₁·Distância + β₂·Peso + β₃·Paradas

> **Novidade desta aula:** além de treinar o modelo, vamos **diagnosticar** se ele atende aos pressupostos da regressão linear.
> Os 4 gráficos de resíduos são a ferramenta padrão para isso na prática profissional.

| Pressuposto | O que verificar |
|-------------|----------------|
| Linearidade | Resíduos aleatórios em torno de zero |
| Normalidade | Resíduos seguem distribuição normal |
| Homocedasticidade | Variância constante dos resíduos |
| Independência | Sem padrões nos resíduos |

In [ ]:
# ── Dados de logística ────────────────────────────────────────
n = 200
distancia_km = np.random.uniform(10, 500, n)
peso_kg      = np.random.uniform(0.5, 50, n)
paradas      = np.random.randint(0, 6, n)
ruido1       = np.random.normal(0, 1.5, n)

# Fórmula real: tempo = 2 + 0.05·dist + 0.3·peso + 1.5·paradas + ruído
tempo_horas = 2 + 0.05 * distancia_km + 0.3 * peso_kg + 1.5 * paradas + ruido1

df1 = pd.DataFrame({
    'distancia_km': distancia_km.round(1),
    'peso_kg':      peso_kg.round(1),
    'paradas':      paradas,
    'tempo_horas':  tempo_horas.round(2)
})

print('Estatísticas descritivas:')
print(df1.describe().round(2))
print('\nCorrelação com tempo_horas:')
print(df1.corr()[['tempo_horas']].round(3))

In [ ]:
# ── Exploração visual ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
pares = [('distancia_km', '#0d9488'), ('peso_kg', '#2563eb'), ('paradas', '#e11d48')]

for ax, (col, cor) in zip(axes, pares):
    ax.scatter(df1[col], df1['tempo_horas'], color=cor, alpha=.4, s=25)
    r = df1[col].corr(df1['tempo_horas'])
    ax.set_xlabel(col.replace('_', ' ').title())
    ax.set_ylabel('Tempo (horas)')
    ax.set_title(f'Correlação r = {r:.3f}')

plt.suptitle('Exemplo 1 — Relação entre preditores e tempo de entrega', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Treinar e avaliar ─────────────────────────────────────────
X1 = df1[['distancia_km', 'peso_kg', 'paradas']]
y1 = df1['tempo_horas']

X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.2, random_state=42
)

modelo1 = LinearRegression()
modelo1.fit(X1_train, y1_train)
y1_pred = modelo1.predict(X1_test)

coef_df = pd.DataFrame({
    'Variável':    ['intercepto'] + list(X1.columns),
    'β estimado':  [round(modelo1.intercept_, 3)] + [round(c, 3) for c in modelo1.coef_],
    'β real':      [2.0, 0.05, 0.3, 1.5]
})
print('Coeficientes estimados vs reais:')
print(coef_df.to_string(index=False))

mae  = mean_absolute_error(y1_test, y1_pred)
rmse = np.sqrt(mean_squared_error(y1_test, y1_pred))
r2   = r2_score(y1_test, y1_pred)

print(f'\nMétricas no conjunto de teste:')
print(f'  MAE  = {mae:.3f} h  (erro médio de {mae*60:.0f} min)')
print(f'  RMSE = {rmse:.3f} h')
print(f'  R²   = {r2:.4f}  ({r2*100:.1f}% da variação explicada)')
print(f'\nInterpretação (mantendo as demais constantes):')
print(f'  +1 km   → +{modelo1.coef_[0]*60:.1f} min de entrega')
print(f'  +1 kg   → +{modelo1.coef_[1]*60:.0f} min de entrega')
print(f'  +1 parada → +{modelo1.coef_[2]:.1f} h de entrega')

In [ ]:
# ── 4 Gráficos de Diagnóstico dos Resíduos ────────────────────
residuos = y1_test.values - y1_pred

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# 1. Resíduos vs Valores Ajustados
axes[0, 0].scatter(y1_pred, residuos, color='#2563eb', alpha=.6, s=30)
axes[0, 0].axhline(0, color='red', lw=1.5, ls='--')
axes[0, 0].set_xlabel('Valores Ajustados (ŷ)')
axes[0, 0].set_ylabel('Resíduo (y − ŷ)')
axes[0, 0].set_title('1. Resíduos vs Valores Ajustados\n(ideal: nuvem aleatória em torno de zero)')
axes[0, 0].grid(alpha=.3)

# 2. Q-Q Plot (normalidade dos resíduos)
(osm, osr), (slope, intercept, _) = stats.probplot(residuos, dist='norm')
axes[0, 1].scatter(osm, osr, color='#0d9488', s=20, alpha=.7)
x_line = np.array([osm.min(), osm.max()])
axes[0, 1].plot(x_line, slope * x_line + intercept, color='red', lw=1.5)
axes[0, 1].set_xlabel('Quantis teóricos (Normal)')
axes[0, 1].set_ylabel('Quantis dos resíduos')
axes[0, 1].set_title('2. Q-Q Plot\n(ideal: pontos na linha = distribuição normal)')
axes[0, 1].grid(alpha=.3)

# 3. Histograma dos resíduos
axes[1, 0].hist(residuos, bins=18, color='#6366f1', edgecolor='white', alpha=.8)
axes[1, 0].axvline(0, color='red', ls='--', lw=1.5, label='Zero')
axes[1, 0].axvline(residuos.mean(), color='#d97706', ls='-', lw=1.5,
                    label=f'Média = {residuos.mean():.3f}')
axes[1, 0].set_xlabel('Resíduo')
axes[1, 0].set_ylabel('Frequência')
axes[1, 0].set_title(f'3. Distribuição dos Resíduos\nDesvio-padrão = {residuos.std():.3f}')
axes[1, 0].legend()

# 4. Real vs Previsto
diag = np.linspace(y1_test.min(), y1_test.max(), 50)
axes[1, 1].scatter(y1_test, y1_pred, color='#e11d48', alpha=.6, s=30)
axes[1, 1].plot(diag, diag, 'k--', lw=1.5, label='Predição perfeita')
axes[1, 1].set_xlabel('Valor Real (horas)')
axes[1, 1].set_ylabel('Valor Previsto (horas)')
axes[1, 1].set_title(f'4. Real vs Previsto\nR² = {r2:.4f}')
axes[1, 1].legend()

plt.suptitle('Diagnóstico da Regressão Linear — 4 Gráficos Essenciais',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Teste de normalidade dos resíduos
stat_sw, p_sw = stats.shapiro(residuos)
print(f'Teste de Shapiro-Wilk: estatística = {stat_sw:.4f} | p-valor = {p_sw:.4f}')
print(f'→ Resíduos {"seguem" if p_sw > 0.05 else "NÃO seguem"} '
      f'distribuição normal (α = 0,05)')

---
## 2 · Exemplo 2 — Regressão Logística: ROC, AUC e Limiar
### Problema: prever aprovação ou reprovação de alunos

**Variável alvo (Y):** Aprovado (1) ou Reprovado (0)  
**Preditores (X):** Frequência (%), Nota parcial (0–10), Horas de estudo/semana  
**Saída:** Probabilidade de aprovação entre 0 e 1

> **Novidade desta aula:** na Aula 09 usamos o limiar padrão de 0,5 para classificar.
> Aqui aprendemos a **escolher o melhor limiar** via curva ROC e a quantificar a qualidade do modelo com a **AUC** (Area Under the Curve).

| Métrica | Significado |
|---------|-------------|
| **Acurácia** | % de acertos totais |
| **Precisão** | Dos que previmos como aprovados, quantos realmente passaram? |
| **Recall** | Dos que realmente passaram, quantos o modelo identificou? |
| **F1-Score** | Média harmônica entre precisão e recall |
| **AUC** | Capacidade geral de separar as classes (0,5 = aleatório; 1,0 = perfeito) |

In [ ]:
# ── Dados de alunos ───────────────────────────────────────────
n2 = 300
frequencia_pct  = np.random.uniform(50, 100, n2)
nota_parcial    = np.random.uniform(2.0, 10.0, n2)
horas_estudo    = np.random.uniform(0, 20, n2)

# Probabilidade real de aprovação
logit2   = -8 + 0.06 * frequencia_pct + 0.6 * nota_parcial + 0.08 * horas_estudo
prob2    = 1 / (1 + np.exp(-logit2))
aprovado = (np.random.uniform(0, 1, n2) < prob2).astype(int)

df2 = pd.DataFrame({
    'frequencia_pct':   frequencia_pct.round(1),
    'nota_parcial':     nota_parcial.round(2),
    'horas_estudo_sem': horas_estudo.round(1),
    'aprovado':         aprovado
})

print('Distribuição das classes:')
print(df2['aprovado'].value_counts().rename({0: 'Reprovado (0)', 1: 'Aprovado (1)'}))
print(f'\nTaxa de aprovação: {df2["aprovado"].mean():.1%}')
df2.head(6)

In [ ]:
# ── Treinar regressão logística ───────────────────────────────
X2 = df2[['frequencia_pct', 'nota_parcial', 'horas_estudo_sem']]
y2 = df2['aprovado']

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.25, random_state=42, stratify=y2
)

scaler2      = StandardScaler()
X2_train_s   = scaler2.fit_transform(X2_train)
X2_test_s    = scaler2.transform(X2_test)

modelo2 = LogisticRegression(random_state=42, max_iter=500)
modelo2.fit(X2_train_s, y2_train)

y2_pred  = modelo2.predict(X2_test_s)
y2_proba = modelo2.predict_proba(X2_test_s)[:, 1]

print('Relatório de classificação (limiar = 0,5):')
print(classification_report(y2_test, y2_pred, target_names=['Reprovado', 'Aprovado']))

# Odds ratios — interpretação dos coeficientes
odds_ratios = np.exp(modelo2.coef_[0])
print('Odds Ratios (impacto de cada variável padronizada):')
for var, odds in zip(X2.columns, odds_ratios):
    direcao = 'aumenta' if odds > 1 else 'diminui'
    print(f'  {var:25s}: OR = {odds:.3f}  → {direcao} a chance de aprovação')

In [ ]:
# ── Função sigmoide + distribuição de probabilidades ─────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Função sigmoide
z = np.linspace(-7, 7, 300)
sig = 1 / (1 + np.exp(-z))
axes[0].plot(z, sig, color='#2563eb', lw=2.5)
axes[0].axhline(0.5, color='red', ls='--', lw=1.2, label='Limiar = 0,5')
axes[0].axvline(0, color='gray', ls=':', lw=1)
axes[0].fill_between(z, sig, 0.5, where=sig > 0.5, alpha=.15, color='#0d9488', label='Aprovado')
axes[0].fill_between(z, sig, 0.5, where=sig < 0.5, alpha=.15, color='#e11d48', label='Reprovado')
axes[0].set_xlabel('Combinação linear dos preditores (z)')
axes[0].set_ylabel('Probabilidade de aprovação')
axes[0].set_title('Função Sigmoide\n(transforma qualquer número em probabilidade 0–1)')
axes[0].legend()
axes[0].grid(alpha=.3)

# Distribuição das probabilidades previstas por classe real
for cls, cor, nome in [(0, '#e11d48', 'Reprovado'), (1, '#0d9488', 'Aprovado')]:
    mask = y2_test.values == cls
    axes[1].hist(y2_proba[mask], bins=20, alpha=.65, color=cor,
                 label=nome, edgecolor='white')
axes[1].axvline(0.5, color='black', ls='--', lw=2, label='Limiar = 0,5')
axes[1].set_xlabel('Probabilidade prevista de aprovação')
axes[1].set_ylabel('Frequência')
axes[1].set_title('Distribuição das probabilidades\npor classe real')
axes[1].legend()

plt.suptitle('Exemplo 2 — Regressão Logística: função sigmoide e probabilidades',
             fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Curva ROC, AUC e análise do limiar ────────────────────────
fpr, tpr, thresholds_roc = roc_curve(y2_test, y2_proba)
auc_score = roc_auc_score(y2_test, y2_proba)

# Melhor limiar pelo índice de Youden (maximiza TPR - FPR)
j_idx    = np.argmax(tpr - fpr)
best_thr = thresholds_roc[j_idx]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Curva ROC
axes[0].plot(fpr, tpr, color='#2563eb', lw=2.5, label=f'Modelo (AUC = {auc_score:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Aleatório (AUC = 0,5)')
axes[0].fill_between(fpr, tpr, alpha=.1, color='#2563eb')
axes[0].scatter(fpr[j_idx], tpr[j_idx], color='red', s=120, zorder=5,
                label=f'Limiar ótimo = {best_thr:.2f}')
axes[0].set_xlabel('Taxa de Falso Positivo (FPR = 1 − Especificidade)')
axes[0].set_ylabel('Taxa de Verdadeiro Positivo (TPR = Recall)')
axes[0].set_title('Curva ROC\n(quanto mais perto do canto sup. esq., melhor)')
axes[0].legend()
axes[0].grid(alpha=.3)

# Métricas em função do limiar
thrs = np.linspace(0.05, 0.95, 60)
precs, recs, f1s, accs = [], [], [], []
for thr in thrs:
    y_thr = (y2_proba >= thr).astype(int)
    precs.append(precision_score(y2_test, y_thr, zero_division=0))
    recs.append(recall_score(y2_test, y_thr, zero_division=0))
    f1s.append(f1_score(y2_test, y_thr, zero_division=0))
    accs.append(accuracy_score(y2_test, y_thr))

axes[1].plot(thrs, precs, color='#0d9488', lw=2, label='Precisão')
axes[1].plot(thrs, recs,  color='#e11d48', lw=2, label='Recall')
axes[1].plot(thrs, f1s,   color='#d97706', lw=2, label='F1-Score')
axes[1].plot(thrs, accs,  color='#6366f1', lw=2, ls='--', label='Acurácia')
axes[1].axvline(0.5,      color='gray',  ls=':', lw=1.5, label='Limiar padrão 0,5')
axes[1].axvline(best_thr, color='red',   ls='--', lw=1.5,
                label=f'Limiar ótimo {best_thr:.2f}')
axes[1].set_xlabel('Limiar de decisão')
axes[1].set_ylabel('Valor da métrica')
axes[1].set_title('Trade-off: precisão × recall\n(escolher o limiar é uma decisão de negócio)')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=.3)

plt.suptitle('Exemplo 2 — Curva ROC e Otimização do Limiar', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'AUC = {auc_score:.4f}')
print(f'Limiar ótimo (Youden): {best_thr:.3f}')
y2_pred_opt = (y2_proba >= best_thr).astype(int)
print(f'\nComparativo — limiar 0,50 vs {best_thr:.2f}:')
print(f'  Acurácia:   {accuracy_score(y2_test, y2_pred):.3f}  →  {accuracy_score(y2_test, y2_pred_opt):.3f}')
print(f'  F1-Score:   {f1_score(y2_test, y2_pred):.3f}  →  {f1_score(y2_test, y2_pred_opt):.3f}')

---
## 3 · Exemplo 3 — Validação Cruzada: Avaliação Justa e Robusta
### Por que um único split treino/teste pode enganar?

> **Problema:** ao dividir os dados uma única vez, o resultado depende de **quais amostras caíram no teste**.
> Se os dados de teste foram fáceis, o modelo parece ótimo. Se foram difíceis, parece ruim.
> A **validação cruzada k-fold** resolve isso: divide os dados em k partes e testa o modelo em cada uma delas.

```
K-Fold com k=5:

Fold 1: [TESTE] [treino] [treino] [treino] [treino]
Fold 2: [treino] [TESTE] [treino] [treino] [treino]
Fold 3: [treino] [treino] [TESTE] [treino] [treino]
Fold 4: [treino] [treino] [treino] [TESTE] [treino]
Fold 5: [treino] [treino] [treino] [treino] [TESTE]
                                                   ↓
                              Média e desvio-padrão das 5 métricas
```

**Vantagem:** todos os dados são usados para treino E para teste (em momentos diferentes).

In [ ]:
# ── Comparação: split simples vs validação cruzada ────────────
pipe_cv = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(random_state=42, max_iter=500))
])

resultados_cv = {}

# Split simples repetido 10 vezes (mostra a instabilidade)
simples = []
for seed in range(10):
    Xtr, Xte, ytr, yte = train_test_split(
        X2, y2, test_size=0.25, random_state=seed, stratify=y2
    )
    sc = StandardScaler()
    m  = LogisticRegression(random_state=42, max_iter=500)
    m.fit(sc.fit_transform(Xtr), ytr)
    simples.append(roc_auc_score(yte, m.predict_proba(sc.transform(Xte))[:, 1]))
resultados_cv['Split simples\n(10 repetições)'] = np.array(simples)

# K-Fold com k = 3, 5, 10
for k in [3, 5, 10]:
    skf    = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    scores = cross_val_score(pipe_cv, X2, y2, cv=skf, scoring='roc_auc')
    resultados_cv[f'CV {k}-fold'] = scores

print('AUC por método de validação:')
print(f'{"Método":30s} {"Média":>7} {"Desvio":>8} {"Min":>7} {"Max":>7}')
print('-' * 65)
for nome, scores in resultados_cv.items():
    nome_limpo = nome.replace('\n', ' ')
    print(f'{nome_limpo:30s} {scores.mean():>7.4f} {scores.std():>8.4f} '
          f'{scores.min():>7.4f} {scores.max():>7.4f}')

fig, ax = plt.subplots(figsize=(9, 4))
nomes   = list(resultados_cv.keys())
valores = [resultados_cv[n] for n in nomes]
labels  = [n.replace('\n', ' ') for n in nomes]

bp = ax.boxplot(valores, labels=labels, patch_artist=True, widths=0.5,
                boxprops=dict(facecolor='#dbeafe'),
                medianprops=dict(color='#1d4ed8', lw=2))
ax.set_ylabel('AUC (Área sob a curva ROC)')
ax.set_ylim(0.65, 1.02)
ax.set_title('Validação Cruzada vs Split Simples\n'
             '(caixa menor = estimativa mais estável)')
ax.grid(axis='y', alpha=.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Curva de Aprendizado ──────────────────────────────────────
# Mostra como o modelo melhora à medida que recebe mais dados de treino

train_sizes, train_scores, val_scores = learning_curve(
    pipe_cv, X2, y2,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='roc_auc'
)

tr_mean = train_scores.mean(axis=1)
tr_std  = train_scores.std(axis=1)
va_mean = val_scores.mean(axis=1)
va_std  = val_scores.std(axis=1)
gap     = tr_mean - va_mean

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Curva de aprendizado
axes[0].plot(train_sizes, tr_mean, 'o-', color='#2563eb', lw=2, label='Treino')
axes[0].fill_between(train_sizes, tr_mean - tr_std, tr_mean + tr_std,
                     alpha=.15, color='#2563eb')
axes[0].plot(train_sizes, va_mean, 's-', color='#e11d48', lw=2, label='Validação (CV)')
axes[0].fill_between(train_sizes, va_mean - va_std, va_mean + va_std,
                     alpha=.15, color='#e11d48')
axes[0].set_xlabel('Amostras de treinamento')
axes[0].set_ylabel('AUC')
axes[0].set_title('Curva de Aprendizado\n'
                  'Gap grande = overfitting | Ambas baixas = underfitting')
axes[0].legend()
axes[0].grid(alpha=.3)
axes[0].set_ylim(0.5, 1.05)

# Gap treino vs validação ao longo do tamanho
axes[1].bar(range(len(train_sizes)), gap, color='#6366f1', alpha=.8)
axes[1].set_xticks(range(len(train_sizes)))
axes[1].set_xticklabels([f'{int(s)}' for s in train_sizes], rotation=45)
axes[1].axhline(0.05, color='orange', ls='--', lw=1.5, label='Referência 0,05')
axes[1].set_xlabel('Amostras de treinamento')
axes[1].set_ylabel('Gap (AUC treino − AUC validação)')
axes[1].set_title('Gap de Overfitting por Tamanho\n(ideal: gap próximo de zero)')
axes[1].legend()
axes[1].grid(axis='y', alpha=.3)

plt.suptitle('Exemplo 3 — Curva de Aprendizado', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Gap final (treino vs validação): {gap[-1]:.4f}')
risco = 'Baixo' if gap[-1] < 0.05 else 'Moderado' if gap[-1] < 0.10 else 'Alto'
print(f'→ {risco} risco de overfitting com os dados completos')

---
## 4 · Exemplo 4 — Overfitting vs Underfitting
### A complexidade certa faz toda a diferença

> **Underfitting:** modelo simples demais — não captura o padrão dos dados (alto erro no treino *e* no teste).
> **Overfitting:** modelo complexo demais — decora o treino mas não generaliza (baixo erro no treino, alto no teste).

Vamos usar **regressão polinomial** para visualizar isso de forma clara:
- Grau 1 → reta simples (underfitting)
- Grau 3 → ajuste razoável
- Grau 7 → começa a overfittar
- Grau 12 → overfitting severo

In [ ]:
# ── Dados para demonstração ───────────────────────────────────
np.random.seed(7)
n_ov   = 35
X_ov   = np.random.uniform(0, 10, n_ov)
y_ov   = 2 * np.sin(X_ov * 0.8) + np.random.normal(0, 0.8, n_ov)

X_ov_tr, X_ov_te, y_ov_tr, y_ov_te = train_test_split(
    X_ov, y_ov, test_size=0.3, random_state=42
)
x_plot = np.linspace(0, 10, 400)

graus  = [1, 3, 7, 12]
cores  = ['#0d9488', '#2563eb', '#d97706', '#e11d48']
status = ['Underfitting', 'Bom ajuste', 'Início do overfitting', 'Overfitting severo']

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
resumo_ov = []

for ax, grau, cor, sts in zip(axes.flatten(), graus, cores, status):
    pipe_ov = Pipeline([
        ('poly',   PolynomialFeatures(degree=grau, include_bias=False)),
        ('scaler', StandardScaler()),
        ('model',  LinearRegression())
    ])
    pipe_ov.fit(X_ov_tr.reshape(-1, 1), y_ov_tr)

    r2_tr = r2_score(y_ov_tr, pipe_ov.predict(X_ov_tr.reshape(-1, 1)))
    r2_te = r2_score(y_ov_te, pipe_ov.predict(X_ov_te.reshape(-1, 1)))
    resumo_ov.append({'Grau': grau, 'R² Treino': r2_tr, 'R² Teste': r2_te, 'Status': sts})

    y_curva = pipe_ov.predict(x_plot.reshape(-1, 1))

    ax.scatter(X_ov_tr, y_ov_tr, color='#475569', s=55, zorder=3, label='Treino')
    ax.scatter(X_ov_te, y_ov_te, color='black',   s=70, marker='D', zorder=3, label='Teste')
    ax.plot(x_plot, y_curva, color=cor, lw=2.5, label=f'Grau {grau}')
    ax.set_xlim(-0.5, 10.5)
    ax.set_ylim(-5, 5)
    ax.set_title(f'Grau {grau} — {sts}\nR²_treino={r2_tr:.3f} | R²_teste={r2_te:.3f}',
                 color=cor, fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Overfitting com Regressão Polinomial\n'
             'Quanto mais complexo, mais "memoriza" o treino e perde generalização',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

df_ov = pd.DataFrame(resumo_ov)
print(df_ov.round(4).to_string(index=False))

In [ ]:
# ── Curva Bias-Variance: erro de treino vs teste por grau ─────
graus_seq = list(range(1, 13))
r2_treino, r2_teste = [], []

for g in graus_seq:
    p = Pipeline([
        ('poly',   PolynomialFeatures(degree=g, include_bias=False)),
        ('scaler', StandardScaler()),
        ('model',  LinearRegression())
    ])
    p.fit(X_ov_tr.reshape(-1, 1), y_ov_tr)
    r2_treino.append(r2_score(y_ov_tr, p.predict(X_ov_tr.reshape(-1, 1))))
    r2_teste.append(r2_score( y_ov_te, p.predict(X_ov_te.reshape(-1, 1))))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(graus_seq, r2_treino, 'o-', color='#2563eb', lw=2, label='R² no Treino')
ax.plot(graus_seq, r2_teste,  's-', color='#e11d48', lw=2, label='R² no Teste')
ax.axvline(3, color='gray', ls=':', lw=1.5, label='Grau ideal ≈ 3')

ax.annotate('Underfitting\n(ambos baixos)', xy=(1, r2_treino[0]),
            xytext=(2.2, 0.1), fontsize=9, color='#0d9488',
            arrowprops=dict(arrowstyle='->', color='#0d9488'))
ax.annotate('Overfitting\n(treino alto,\nteste cai)', xy=(9, r2_teste[8]),
            xytext=(7, -0.5), fontsize=9, color='#d97706',
            arrowprops=dict(arrowstyle='->', color='#d97706'))

ax.set_xlabel('Grau do polinômio (complexidade do modelo)')
ax.set_ylabel('R²')
ax.set_title('Tradeoff Bias-Variância\n'
             'Existe um ponto ótimo entre complexidade e generalização')
ax.legend()
ax.grid(alpha=.3)
ax.set_xticks(graus_seq)
plt.tight_layout()
plt.show()

---
## 5 · Comparativo Final de Modelos

Reunindo os resultados dos exemplos anteriores em um painel comparativo.

In [ ]:
# ── Painel comparativo ────────────────────────────────────────
comp = pd.DataFrame([
    {'Modelo': 'Reg. Linear (entrega)',
     'Tipo': 'Regressão', 'Métrica': 'R²',
     'Valor': r2_score(y1_test, y1_pred)},
    {'Modelo': 'Reg. Logística — limiar 0,50',
     'Tipo': 'Classificação', 'Métrica': 'F1',
     'Valor': f1_score(y2_test, y2_pred)},
    {'Modelo': f'Reg. Logística — limiar ótimo {best_thr:.2f}',
     'Tipo': 'Classificação', 'Métrica': 'F1',
     'Valor': f1_score(y2_test, (y2_proba >= best_thr).astype(int))},
    {'Modelo': 'Reg. Logística — AUC (CV 5-fold)',
     'Tipo': 'Classificação', 'Métrica': 'AUC',
     'Valor': resultados_cv['CV 5-fold'].mean()},
])

print('Comparativo de resultados:')
print(comp.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
cores_comp = ['#0d9488', '#2563eb', '#15803d', '#6366f1']
bars = ax.barh(comp['Modelo'], comp['Valor'],
               color=cores_comp, height=0.55, edgecolor='white')
ax.set_xlabel('Valor da métrica (R², F1 ou AUC)')
ax.set_xlim(0, 1.1)
for bar, val in zip(bars, comp['Valor']):
    ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=10)
ax.set_title('Comparativo Final — Métricas por Modelo', fontweight='bold')
plt.tight_layout()
plt.show()

print('\nInsights:')
print('  • Escolher o limiar adequado melhorou o F1 da Regressão Logística')
print('  • A validação cruzada (CV) fornece uma estimativa mais confiável')
print('  • Sempre compare modelos com a mesma métrica e o mesmo conjunto de dados')

---
## 6 · EXERCÍCIO — Sua Vez!

### Contexto
Um hospital público quer prever quais pacientes têm maior probabilidade de **reinternação em 30 dias** após a alta.
Identificar esses pacientes com antecedência permite alocar acompanhamento especial e reduzir custos.

| Variável | Descrição |
|----------|-----------|
| `idade` | Idade do paciente (anos) |
| `internacoes_prev` | Internações hospitalares nos últimos 12 meses |
| `dias_internado` | Dias de internação no episódio atual |
| `doencas_cronicas` | Número de doenças crônicas diagnosticadas |
| `reinternado` | 1 = reinternado em 30 dias; 0 = não reinternado |

### Tarefas

1. **Explore os dados** — estatísticas descritivas e distribuição das classes
2. **Treine uma Regressão Logística** — divida os dados e padronize os preditores
3. **Calcule as métricas** — acurácia, precisão, recall, F1 e AUC
4. **Plote a Curva ROC** — identifique o limiar ótimo
5. **Aplique validação cruzada** — compare com o resultado do split simples
6. **Interprete:** que variável mais impacta a reinternação? O modelo é confiável para uso clínico?

> **Dica:** use como referência os Exemplos 2 e 3 desta aula.

In [ ]:
# ── Dados do exercício (não altere esta célula) ────────────────
np.random.seed(99)
n_ex = 400

idade_ex         = np.random.randint(18, 91, n_ex)
internacoes_prev = np.random.randint(0, 6, n_ex)
dias_internado   = np.random.randint(1, 31, n_ex)
doencas_cronicas = np.random.randint(0, 6, n_ex)

logit_ex    = (-3.5
               + 0.025 * idade_ex
               + 0.55  * internacoes_prev
               + 0.04  * dias_internado
               + 0.45  * doencas_cronicas)
prob_ex     = 1 / (1 + np.exp(-logit_ex))
reinternado = (np.random.uniform(0, 1, n_ex) < prob_ex).astype(int)

df_ex = pd.DataFrame({
    'idade':            idade_ex,
    'internacoes_prev': internacoes_prev,
    'dias_internado':   dias_internado,
    'doencas_cronicas': doencas_cronicas,
    'reinternado':      reinternado
})

print('Dados do exercício — primeiras linhas:')
print(df_ex.head(10))
print(f'\nTotal de pacientes: {len(df_ex)}')

In [ ]:
# ── TAREFA 1: Exploração dos dados ────────────────────────────
# Use .describe() e verifique a distribuição de classes

# Seu código aqui:


In [ ]:
# ── TAREFAS 2 e 3: Regressão Logística + Métricas ─────────────
# Separe X e y, faça split estratificado, padronize, treine e avalie

X_ex = df_ex.drop(columns='reinternado')
y_ex = df_ex['reinternado']

# 1) Divida os dados (use test_size=0.25, stratify=y_ex, random_state=42)
X_ex_train, X_ex_test, y_ex_train, y_ex_test = train_test_split(
    X_ex, y_ex, test_size=_____,  # <-- complete
    random_state=42, stratify=y_ex
)

# 2) Padronize os preditores
scaler_ex    = StandardScaler()
X_ex_train_s = scaler_ex.fit_transform(X_ex_train)
X_ex_test_s  = scaler_ex.transform(X_ex_test)

# 3) Treine a Regressão Logística
modelo_ex = LogisticRegression(random_state=42, max_iter=500)
modelo_ex.fit(_____, _____)  # <-- complete

# 4) Previsões
y_ex_pred  = modelo_ex.predict(_____)
y_ex_proba = modelo_ex.predict_proba(_____)[:, 1]

# 5) Métricas
print('Relatório de classificação:')
print(classification_report(y_ex_test, y_ex_pred,
                             target_names=['Não reinternado', 'Reinternado']))
print(f'AUC = {roc_auc_score(y_ex_test, y_ex_proba):.4f}')

In [ ]:
# ── TAREFA 4: Curva ROC ───────────────────────────────────────
# Plote a curva ROC e identifique o limiar ótimo (Youden)

# Seu código aqui:


In [ ]:
# ── TAREFA 5: Validação Cruzada ───────────────────────────────
# Use 5-fold estratificado e compare com o split simples
# Dica: use Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(...))])

# Seu código aqui:


In [ ]:
# ── TAREFA EXTRA: Odds Ratios + Visualização ──────────────────
# Crie um gráfico de barras com os odds ratios de cada variável
# Qual variável tem maior impacto na reinternação?

# Seu código aqui:


---
### Perguntas para reflexão

Responda nas células abaixo (texto livre — clique duas vezes para editar):

**1.** Qual variável mais impacta a reinternação? O resultado faz sentido clinicamente?

> *Sua resposta aqui...*

**2.** Para um hospital, é pior errar classificando um paciente de risco como seguro (*falso negativo*) ou um paciente seguro como de risco (*falso positivo*)? Como isso muda a escolha do limiar?

> *Sua resposta aqui...*

**3.** O resultado da validação cruzada foi muito diferente do split simples? O que isso indica sobre a estabilidade do modelo?

> *Sua resposta aqui...*

**4.** Um modelo com AUC = 0,75 é confiável para uso clínico? O que mais precisaríamos considerar antes de implantar esse sistema em um hospital?

> *Sua resposta aqui...*

---
**Ciência de Dados · Aula 10 · Prof. Mesc. Diego Ramos Inácio · Univassouras · 2026**